# Episode 28: Cost Optimization

You pay per token. And every turn of a conversation re-sends the *entire history* to the model — so a long chat doesn't cost a flat rate per turn, it costs **more every turn**.

**In this episode you'll build:** context controls that keep that growth in check — and you'll measure the savings in tokens.

## Where the cost goes

An LLM is stateless. To answer turn 10, the model must be sent turns 1–9 again. History grows every turn, and so does the prompt you pay for:

```
turn 1:  [system + msg1]                       → small
turn 5:  [system + msg1..msg4 + replies]        → bigger
turn 20: [system + msg1..msg19 + replies]       → expensive
```

Two middleware fix this by **trimming history before each model call**:

- **`HistoryLimiter(max_events=N)`** — keep the last N events. Simple, deterministic.
- **`TokenLimiter(max_tokens=N)`** — keep history under an estimated token budget.

## Measuring the savings

Below we run the *same* 7-turn conversation three times — no limiter, then each limiter — and record the prompt-token count on the **final turn**. That final turn is where an unbounded history hurts most.

We read the count straight off each `ModelResponse` event with an observer.

In [1]:
from dotenv import load_dotenv

load_dotenv()

from autogen.beta import Agent, observer
from autogen.beta.config import OpenAIConfig
from autogen.beta.events import ModelResponse
from autogen.beta.middleware import HistoryLimiter, TokenLimiter

config = OpenAIConfig(model="gpt-4.1-mini", temperature=0)


async def run(label: str, middleware: list | None) -> None:
    last = {"prompt": 0}

    @observer(ModelResponse)
    def capture(event: ModelResponse) -> None:
        last["prompt"] = event.usage.prompt_tokens

    extra = {"middleware": middleware} if middleware else {}
    agent = Agent("chatter", prompt="You are concise. One sentence answers.",
                  config=config, observers=[capture], **extra)

    reply = await agent.ask("Tell me a fact about the number 1.")
    for n in range(2, 8):                       # six follow-up turns
        reply = await reply.ask(f"Now a fact about the number {n}.")

    print(f"{label} -> {last['prompt']} prompt tokens on the final turn")


await run("no limiter            ", None)
await run("HistoryLimiter(max=6) ", [HistoryLimiter(max_events=6)])
await run("TokenLimiter(max=400) ", [TokenLimiter(max_tokens=400)])


no limiter             -> 253 prompt tokens on the final turn


HistoryLimiter(max=6)  -> 137 prompt tokens on the final turn


TokenLimiter(max=400)  -> 200 prompt tokens on the final turn


## Compaction: keep the meaning, drop the bulk

Limiters *delete* old events — cheap, but the agent forgets. **Compaction** is smarter: it replaces a span of old events with a short summary, so the agent keeps context at a fraction of the tokens.

Two built-in strategies in `autogen.beta.compact`:

```python
from autogen.beta.compact import TailWindowCompact, SummarizeCompact

# Zero LLM cost — just keep the last 50 events, drop the rest.
compact = TailWindowCompact(target=50)

# One LLM call — summarize the dropped span into a CompactionSummary event.
compact = SummarizeCompact(target=50, config=config)

retained = await compact.compact(events, ctx, store)
```

A `CompactTrigger` declares *when* to fire:

```python
from autogen.beta.compact import CompactTrigger

trigger = CompactTrigger(max_tokens=32_000)   # compact once history passes 32k tokens
```

Use `TailWindowCompact` when only recent context matters; use `SummarizeCompact` when older context still carries weight.

## Monitoring spend

Trimming controls cost; **monitoring** tells you when you're near a budget. The built-in `TokenMonitor` observer (Episode 25) tracks cumulative usage and emits `WARNING` / `CRITICAL` alerts as thresholds are crossed:

```python
from autogen.beta.observers import TokenMonitor

agent = Agent(..., observers=[TokenMonitor(warn_threshold=50_000, alert_threshold=100_000)])
```

Pair it with a limiter: the limiter caps per-call cost, the monitor watches the running total.

## Additional content

**Limiter vs compaction.** A limiter runs every call and is free but forgetful. Compaction runs occasionally, may cost one LLM call, and preserves a summary. Many production agents use both — a limiter as a hard ceiling, compaction to stay well under it gracefully.

**`HistoryLimiter` is safe by construction.** It preserves the first `ModelRequest` and won't leave an orphaned tool result at the head of the trimmed history — so trimming never corrupts the conversation.

**Pick a cheaper model.** The biggest lever is often the model itself. `gpt-4.1-mini` (used throughout this workshop) is a fraction of the cost of a frontier model — match the model to the task.

## What's next

That's cost under control. Episode 29 closes Group 6 by **deploying** your agent — exposing it as a service the outside world can call.